# Train segmentation vùng giả mạo trên Kaggle

Trước khi chạy (panel bên phải, mục **Settings**):
- **Accelerator**: GPU T4 x2 (mỗi mô hình trong `RUNS` chạy trên một GPU)
- **Internet**: On (để clone code từ GitHub và tải checkpoint `openmmlab/upernet-swin-tiny`)
- **Input**: `chituok/ffpp-faces-c23` (private); thêm `chituok/ffpp-faces-c40` nếu config có `mix_c40: true`

Bấm **Save Version → Save & Run All (Commit)** để chạy nền (tắt trình duyệt vẫn chạy).
Mỗi phiên tối đa 12 giờ, nên train tự dừng sau khoảng 11 giờ, ở cuối một epoch.
Kết quả nằm ở tab **Output**: `runs/<tên run>/`, log từng mô hình ở `<tên run>.log`.

**Train tiếp ở phiên sau:** tải `last.pt`, `best.pt`, `history.jsonl` của phiên trước lên
một dataset riêng tư **tên mới cho mỗi phiên** (vd `b1-mix-ckpt-e12`; Kaggle giữ bản cũ của
dataset đã gắn vào notebook nên dùng lại tên cũ sẽ train lại từ checkpoint cũ), Add Input
dataset đó, rồi ghi tên dataset vào cột thứ hai của `RUNS`.

In [ ]:
# Mỗi dòng một mô hình: (config, dataset chứa checkpoint để train tiếp, '' = train mới).
# Hai dòng chạy song song, mỗi mô hình một GPU.
RUNS = [
    ('configs/swin_upernet.yaml', ''),
]
EXTRA = ''   # thêm cho mọi mô hình, vd '--epochs 1 --fraction 0.05' để chạy thử

In [ ]:
import csv, glob, os, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor

# /kaggle/input là ổ mạng, mỗi ảnh chờ ~0,25 s nên GPU đứng chờ dữ liệu (~1 s/bước):
# chép một lần sang đĩa máy bằng nhiều luồng song song. Bản c23 vào /tmp/faces, /tmp/index;
# bản c40 vào /tmp/faces_c40, /tmp/index_c40 (đúng tên mà `mix_c40` tìm).
# Chỉ dò 3 tầng đầu: quét đệ quy cả trăm nghìn ảnh trên ổ mạng mất nhiều phút.
found = sorted({f for d in ('*', '*/*', '*/*/*') for f in glob.glob('/kaggle/input/' + d + '/index/train.csv')})
assert found, 'chưa Add Input dataset ffpp-faces-c23'
for f in found:
    index = os.path.dirname(f)
    faces = os.path.join(os.path.dirname(index), 'faces')
    comp = next(csv.DictReader(open(f, encoding='utf-8')))['compression']
    suffix = '' if comp == 'c23' else '_' + comp
    shutil.copytree(index, '/tmp/index' + suffix, dirs_exist_ok=True)
    files = sorted({r[k] for s in ('train', 'val', 'test')
                    for r in csv.DictReader(open(f'{index}/{s}.csv', encoding='utf-8'))
                    for k in ('frame', 'mask') if r[k]})
    def copy(rel, dst_root='/tmp/faces' + suffix, src_root=faces):
        dst = os.path.join(dst_root, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copyfile(os.path.join(src_root, rel), dst)
    t = time.time()
    with ThreadPoolExecutor(128) as ex:
        for i, _ in enumerate(ex.map(copy, files), 1):
            if i % 50000 == 0 or i == len(files):
                print(f'{comp}: chép {i}/{len(files)} ảnh, {time.time() - t:.0f} s', flush=True)
FACES, INDEX = '/tmp/faces', '/tmp/index'
os.environ['TQDM_MININTERVAL'] = '60'   # log thanh tiến trình mỗi phút, tránh log quá dài

In [ ]:
!nvidia-smi -L
!rm -rf /tmp/code && git clone -q https://github.com/nguyenchitu2005-jpg/DoAnChuyenNganh.git /tmp/code
# cùng phiên bản với máy nhà để checkpoint nạp lại được khi đánh giá
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8 transformers==5.19.0

In [ ]:
%cd /tmp/code
import yaml
workers = 4 if len(RUNS) == 1 else 2   # máy Kaggle có 4 nhân CPU, chia cho các mô hình
procs = []
for gpu, (config, resume) in enumerate(RUNS):
    name = yaml.safe_load(open(config))['name']
    run_dir = '/kaggle/working/runs/' + name
    extra = EXTRA
    if resume:   # input chỉ đọc: chép checkpoint sang working/runs/<tên run> rồi train tiếp
        src = [p for d in ('.', '*', '*/*') for p in glob.glob(f'/kaggle/input/{d}/{resume}')][0]
        shutil.copytree(src, run_dir, dirs_exist_ok=True)
        print(name, ': train tiếp từ', src, sorted(os.listdir(run_dir)))
        extra += ' --resume'
    cmd = (f'python src/train.py --config {config} --data-root {FACES} --index-dir {INDEX} '
           f'--num-workers {workers} --runs-dir /kaggle/working/runs --max-hours 11 {extra}')
    log = open(f'/kaggle/working/{name}.log', 'w')
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    procs.append((name, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env)))
    print(f'GPU {gpu}: {name}')

def last_lines(name, n=1):
    text = open(f'/kaggle/working/{name}.log', encoding='utf-8', errors='replace').read()[-20000:]
    lines = [l.strip() for l in text.replace(chr(13), chr(10)).split(chr(10)) if l.strip()]
    return [l for l in lines if 'train tiếp' in l or 'Traceback' in l or 'Error' in l][-2:] + lines[-n:]

# in tiến độ từng mô hình mỗi 5 phút (đọc được bằng `kaggle kernels logs`)
while any(p.poll() is None for _, p in procs):
    time.sleep(300)
    for name, p in procs:
        for line in last_lines(name):
            print(f'[{name}] {line[:200]}', flush=True)
for name, p in procs:
    print(f'[{name}] kết thúc, mã thoát {p.returncode}')
    for line in last_lines(name, 15):
        print(f'[{name}] {line[:200]}')

In [ ]:
# Đánh giá luôn trên FF++ test c23 (c40 và DFD đánh giá ở máy nhà)
for d in glob.glob('/kaggle/working/runs/*'):
    if os.path.exists(d + '/best.pt'):
        !python src/evaluate.py {d}/best.pt --split test